In [1]:
cd /content/drive/MyDrive/Colab Notebooks/music-generator

/content/drive/MyDrive/Colab Notebooks/music-generator


In [2]:
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from music21 import instrument, note, stream, chord
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization

In [3]:
def load_notes():
  with open("data/notes","rb") as f:
    return pickle.load(f)

In [4]:
def build_model(input_shape, vocab_size):
  model=Sequential([
      LSTM(512, return_sequences=True, input_shape=input_shape),
      Dropout(0.3),
      LSTM(512, return_sequences=True),
      Dropout(0.3),
      LSTM(512),
      BatchNormalization(),
      Dense(256, activation='relu'),
      Dropout(0.3),
      Dense(vocab_size, activation='softmax')
  ])
  model.compile(loss="categorical_crossentropy", optimizer="adam")
  return model

In [5]:
def prepare_input(notes, seq_len=100):
    pitch_names = sorted(set(notes))
    note_to_int = {n: i for i, n in enumerate(pitch_names)}

    sequences = []

    for i in range(len(notes) - seq_len):
        seq = notes[i:i + seq_len]
        sequences.append([note_to_int[n] for n in seq])

    network_input = np.reshape(
        sequences,
        (len(sequences), seq_len, 1)
    )

    network_input = network_input / float(len(pitch_names))

    return network_input, pitch_names

In [6]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

In [7]:
def generate_music(model, net_input, pitch_names, temperature=0.9,length=300):
  int_to_note={i: n for i, n in enumerate(pitch_names)}

  start=np.random.randint(0, len(net_input)-1)
  pattern=net_input[start].reshape(-1).tolist()

  output=[]

  for _ in range(length):
    inp=np.reshape(pattern,(1,len(pattern),1))
    pred=model.predict(inp,verbose=0)[0]
    index=sample(pred,temperature)
    output.append(int_to_note[index])

    pattern.append(index)
    pattern=pattern[1:]

  return output

In [8]:
def create_midi(notes, filename="generated.mid"):
    offset = 0
    output_notes = []

    for n in notes:
        if "." in n:
            chord_notes = []
            for x in n.split("."):
                new_note = note.Note(int(x))
                new_note.storedInstrument = instrument.Piano()
                chord_notes.append(new_note)

            new_chord = chord.Chord(chord_notes)
            new_chord.offset = offset
            output_notes.append(new_chord)

        elif n.isdigit():
            new_note = note.Note(int(n))
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        else:
            new_note = note.Note(n)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        offset += 0.5

    stream.Stream(output_notes).write("midi", fp=filename)

In [9]:
%%writefile model.py
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from music21 import instrument, note, stream, chord
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization

def load_notes():
  with open("data/notes","rb") as f:
    return pickle.load(f)

def build_model(input_shape, vocab_size):
  model=Sequential([
      LSTM(512, return_sequences=True, input_shape=input_shape),
      Dropout(0.3),
      LSTM(512, return_sequences=True),
      Dropout(0.3),
      LSTM(512),
      BatchNormalization(),
      Dense(256, activation='relu'),
      Dropout(0.3),
      Dense(vocab_size, activation='softmax')
  ])
  model.compile(loss="categorical_crossentropy", optimizer="adam")
  return model

def prepare_input(notes, seq_len=100):
    pitch_names = sorted(set(notes))
    note_to_int = {n: i for i, n in enumerate(pitch_names)}

    sequences = []

    for i in range(len(notes) - seq_len):
        seq = notes[i:i + seq_len]
        sequences.append([note_to_int[n] for n in seq])

    network_input = np.reshape(
        sequences,
        (len(sequences), seq_len, 1)
    )

    network_input = network_input / float(len(pitch_names))

    return network_input, pitch_names

def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generate_music(model, net_input, pitch_names, temperature=0.9,length=300):
  int_to_note={i: n for i, n in enumerate(pitch_names)}

  start=np.random.randint(0, len(net_input)-1)
  pattern=net_input[start].reshape(-1).tolist()

  output=[]

  for _ in range(length):
    inp=np.reshape(pattern,(1,len(pattern),1))
    pred=model.predict(inp,verbose=0)[0]
    index=sample(pred,temperature)
    output.append(int_to_note[index])

    pattern.append(index)
    pattern=pattern[1:]

  return output

def create_midi(notes, filename="generated.mid"):
    offset = 0
    output_notes = []

    for n in notes:
        if "." in n:
            chord_notes = []
            for x in n.split("."):
                new_note = note.Note(int(x))
                new_note.storedInstrument = instrument.Piano()
                chord_notes.append(new_note)

            new_chord = chord.Chord(chord_notes)
            new_chord.offset = offset
            output_notes.append(new_chord)

        elif n.isdigit():
            new_note = note.Note(int(n))
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        else:
            new_note = note.Note(n)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        offset += 0.5

    stream.Stream(output_notes).write("midi", fp=filename)


Overwriting model.py


In [10]:
!pip install streamlit

In [11]:
%%writefile app.py
import streamlit as st
import numpy as np
import tensorflow as tf
import os

from model import (
    load_notes,
    prepare_input,
    build_model,
    generate_music,
    create_midi
)

st.set_page_config(
    page_title="🎵 AI Music Generator",
    page_icon="🎹",
    layout="centered"
)

st.title("🎵 AI Music Generator")
st.write("Generate piano music using a deep learning LSTM model")

@st.cache_resource
def load_everything():
    notes = load_notes()
    net_input, pitch_names = prepare_input(notes)

    model = build_model(
        input_shape=(net_input.shape[1], net_input.shape[2]),
        vocab_size=len(pitch_names)
    )

    model.load_weights("weights.hdf5", by_name=True, skip_mismatch=True)
    return model, net_input, pitch_names


model, net_input, pitch_names = load_everything()

temperature = st.slider(
    "🎚️ Temperature (Creativity)",
    0.2, 1.5, 0.9, 0.1
)

length = st.slider("🎶 Music Length", 100, 600, 300, 50)

if st.button("🎼 Generate Music"):
    import time
    np.random.seed(int(time.time()))

    with st.spinner("Composing music... 🎶"):
        generated_notes = generate_music(
            model,
            net_input,
            pitch_names,
            temperature
        )

        output_file = "generated.mid"
        create_midi(generated_notes, output_file)

    st.success("🎉 Music generated!")

    with open(output_file, "rb") as f:
        st.download_button(
            "⬇️ Download MIDI",
            f,
            file_name="ai_music.mid",
            mime="audio/midi"
        )


Overwriting app.py


In [12]:
!pip install streamlit music21 tensorflow numpy pandas pyngrok


In [16]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://social-emus-change.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.194.170.18:8501

2026-01-23 07:37:18.382182: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769153838.433813   11446 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769153838.452162   11446 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769153838.487920   11446 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17